In [1]:
import pandas as pd
import numpy as np
import os



In [2]:
# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix



In [3]:
# For imbalance handling
from imblearn.over_sampling import SMOTE

In [4]:
import pandas as pd

# Read all files one by one
df1 = pd.read_csv("Data_of_Attack_Back_Normal.csv")
df1["attack_type"] = "Normal"

df2 = pd.read_csv("Data_of_Attack_Back_BufferOverflow.csv")
df2["attack_type"] = "BufferOverflow"

df3 = pd.read_csv("Data_of_Attack_Back_FTPWrite.csv")
df3["attack_type"] = "FTPWrite"

df4 = pd.read_csv("Data_of_Attack_Back_GuessPassword.csv")
df4["attack_type"] = "GuessPassword"

df5 = pd.read_csv("Data_of_Attack_Back_Neptune.csv")
df5["attack_type"] = "Neptune"

df6 = pd.read_csv("Data_of_Attack_Back_NMap.csv")
df6["attack_type"] = "NMap"

df7 = pd.read_csv("Data_of_Attack_Back_PortSweep.csv")
df7["attack_type"] = "PortSweep"

df8 = pd.read_csv("Data_of_Attack_Back_RootKit.csv")
df8["attack_type"] = "RootKit"

df9 = pd.read_csv("Data_of_Attack_Back_Satan.csv")
df9["attack_type"] = "Satan"

df10 = pd.read_csv("Data_of_Attack_Back_Smurf.csv")
df10["attack_type"] = "Smurf"

df11 = pd.read_csv("Data_of_Attack_Back.csv")
df11["attack_type"] = "Back"

In [5]:
final_df = pd.concat([
    df1, df2, df3, df4, df5,
    df6, df7, df8, df9, df10, df11
], ignore_index=True)

print(final_df.shape)
final_df.head()

(817550, 83)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,0.001.2,0.001.3,0.1.5,0.21,0.1.6,0.22,0.23,0.24,0.25,0.26
0,0.0,0.0,0.0,0.0,0.00215,0.45076,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,0.0,0.0,0.0,0.00162,0.04528,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,0.00236,0.01228,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,0.00233,0.02032,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.0,0.0,0.0,0.00239,0.00486,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# check class distribution
final_df["attack_type"].value_counts()

attack_type
Normal            576710
Neptune           227228
Satan               5019
Smurf               3007
PortSweep           2964
NMap                1554
Back                 968
GuessPassword         53
BufferOverflow        30
RootKit               10
FTPWrite               7
Name: count, dtype: int64

In [7]:
# create binary target column 
final_df["binary_attack"] = final_df["attack_type"].apply(
    lambda x: 0 if x == "Normal" else 1
)

final_df["binary_attack"].value_counts()

# 0 - normal
# 1- attack

binary_attack
0    576710
1    240840
Name: count, dtype: int64

In [8]:
for col in final_df.columns:
    print(f"'{col}'")

'duration'
' protocol_type'
' service'
' flag'
' src_bytes'
' dst_bytes'
' land'
' wrong_fragment'
' urgent'
' hot'
' num_failed_logins'
' logged_in'
' num_compromised'
' root_shell'
' su_attempted'
' num_root'
' num_file_creations'
' num_shells'
' num_access_files'
' num_outbound_cmds'
' is_host_login'
' is_guest_login'
' count'
' srv_count'
' serror_rate'
' srv_error_rate'
' rerror_rate'
' srv_rerror_rate'
' same_srv_rate'
' diff_srv_rate'
' srv_diff_host_rate'
' dst_host_count'
' dst_host_srv_count'
' dst_host_same_srv_rate'
' dst_host_diff_srv_rate'
' dst_host_same_src_port_rate'
' dst_host_srv_diff_host_rate'
' dst_host_serror_rate'
' dst_host_srv_serror_rate'
' dst_host_rerror_rate'
' dst_host_srv_rerror_rate'
'attack_type'
'0.0026'
' 0'
' 0.07'
' 0.3'
' 0.00116'
' 0.00451'
' 0.4'
' 0.5'
' 0.6'
' 0.2'
' 0.7'
' 0.1'
' 0.8'
' 0.9'
' 0.10'
' 0.11'
' 0.1.1'
' 0.12'
' 0.1.2'
' 0.13'
' 0.14'
' 0.1.3'
' 0.001'
' 0.001.1'
' 0.15'
' 0.16'
' 0.17'
' 0.18'
' 0.1.4'
' 0.19'
' 0.20'
' 0.001.2

In [9]:
final_df.columns = final_df.columns.str.strip()

In [10]:
# encoded categorical columns
final_df = pd.get_dummies(final_df, columns=["protocol_type", "service", "flag"])

In [11]:
# seperate features & target
X = final_df.drop(["attack_type", "binary_attack"], axis=1)
y = final_df["binary_attack"]

In [12]:
# train test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [13]:
final_df.isnull().sum().sort_values(ascending=False)

0.22         817543
0.1.6        817543
0.18         817543
0.1.4        817543
0.19         817543
              ...  
flag_0.06         0
flag_0.07         0
flag_0.08         0
flag_0.09         0
flag_0.1          0
Length: 162, dtype: int64

In [14]:
final_df = final_df.fillna(0)

In [15]:
X = final_df.drop(["attack_type", "binary_attack"], axis=1)
y = final_df["binary_attack"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)



In [16]:
print(y_train.value_counts())

binary_attack
0    403697
1    168588
Name: count, dtype: int64


In [17]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42, k_neighbors=1)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_sm).value_counts())

Before SMOTE:
binary_attack
0    403697
1    168588
Name: count, dtype: int64
After SMOTE:
binary_attack
0    403697
1    403697
Name: count, dtype: int64


In [18]:
# -SMOTE- Synthetic Minority Over-sampling Technique
#It is a method used to solve the problem of imbalanced datasets.

 # handle imbalance(using SMOTE)
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_sm).value_counts())

After SMOTE:
binary_attack
0    403697
1    403697
Name: count, dtype: int64


In [19]:
print(y_train.value_counts())

binary_attack
0    403697
1    168588
Name: count, dtype: int64


In [20]:
# train random forest model
model = RandomForestClassifier(random_state=42)

model.fit(X_train_sm, y_train_sm)

y_pred = model.predict(X_test)

In [21]:
# evaluate model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[173011      2]
 [    10  72242]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    173013
           1       1.00      1.00      1.00     72252

    accuracy                           1.00    245265
   macro avg       1.00      1.00      1.00    245265
weighted avg       1.00      1.00      1.00    245265



In [24]:
# multiclass model
y_multi = final_df["attack_type"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_multi,
    test_size=0.3,
    random_state=42,
    stratify=y_multi
)

model_multi = RandomForestClassifier(random_state=42)
model_multi.fit(X_train, y_train)

y_pred_multi = model_multi.predict(X_test)

print(confusion_matrix(y_test, y_pred_multi))
print(classification_report(y_test, y_pred_multi))

[[   291      0      0      0      0      0      0      0      0      0
       0]
 [     0      7      0      0      0      0      2      0      0      0
       0]
 [     0      0      2      0      0      0      0      0      0      0
       0]
 [     0      0      0     15      0      0      1      0      0      0
       0]
 [     0      0      0      0    465      0      1      0      0      0
       0]
 [     0      0      0      0      0  68168      0      0      0      0
       0]
 [     0      0      0      0      0      0 173012      1      0      0
       0]
 [     0      0      0      0      1      1      3    883      0      1
       0]
 [     0      0      0      0      0      0      3      0      0      0
       0]
 [     0      0      0      0      0      0     17      0      0   1489
       0]
 [     0      0      0      0      0      0      3      0      0      0
     899]]


C:\Users\DELL\anaconda3\anaconda imp\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\DELL\anaconda3\anaconda imp\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

          Back       1.00      1.00      1.00       291
BufferOverflow       1.00      0.78      0.88         9
      FTPWrite       1.00      1.00      1.00         2
 GuessPassword       1.00      0.94      0.97        16
          NMap       1.00      1.00      1.00       466
       Neptune       1.00      1.00      1.00     68168
        Normal       1.00      1.00      1.00    173013
     PortSweep       1.00      0.99      1.00       889
       RootKit       0.00      0.00      0.00         3
         Satan       1.00      0.99      0.99      1506
         Smurf       1.00      1.00      1.00       902

      accuracy                           1.00    245265
     macro avg       0.91      0.88      0.89    245265
  weighted avg       1.00      1.00      1.00    245265



C:\Users\DELL\anaconda3\anaconda imp\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Types of Attacks (Just Basic Understanding)



 #DoS

#Buffer Overflow

#RootKit

#Smurf

#Neptune

#NMap

#Port Sweep